# Suite MOD — Modules: schemas as data

A module is a named set of schemas held by an object of the meta-schema `Schemas.Module.Schema`, registered as
`Schemas.Module`. `Modules.module(schemas)` builds one and `Modules.schemas(module)` reads one back, so schemas are
written, read, validated and compared like any other objects. Within a module each schema is written inline, by kind,
and refers by name to the schemas in the module and those registered.

In [ ]:
from mbse.Schemas.Framework import Comparison as C, JSON, Modules, Plain, Proxies, Schemas as S, Validators, YAML
from support import raises, text

store = Proxies.OfStore()

B = store
Int32 = S.OfNative.resolve(lambda n: n.name("Int32").type(int).bits(32))
Size = S.OfNative.resolve(lambda n: n.name("Size").token("ccpp", "size_t").bytes(8))
Phone = S.OfObject.Builder().name("Phone").properties(text("number")).create()
Reach = S.OfUnion.Builder().branches(lambda b: b.name("phone").of(Phone),
                                     lambda b: b.name("other").of(S.OfObject.Builder().properties(text("note")).create())
                                     ).create()
Stamped = S.OfIntersection.Builder().parts(lambda p: p.name("a").of(Phone), lambda p: p.name("b").of(Phone)).create()
Contact = S.OfObject.Builder().name("Contact").ref().properties(
    text("name"), lambda p: p.name("home").of(Phone), lambda p: p.name("reach").of(Reach),
    lambda p: p.name("stamped").of(Stamped), lambda p: p.name("ids").of(lambda t: t.as_indexed(lambda i: i.of(Int32)))
    ).create()
Directory = S.OfObject.Builder().name("Directory").ref().singleton("Directory").create()
Listing = S.OfRelation.Builder().name("Listing").links("directory", "contact").properties(text("key")).unique(
    "directory", "key").unique("contact").create()
Labels = S.OfRelation.Builder().name("Labels").links("contact", "phone").create()  # registered, not in the module, below
S.OfObject.Builder(Contact).relations(lambda a: a.name("listings").of(Listing).me("contact"),
                                      lambda a: a.name("labels").of(Labels).me("contact")).update()
S.OfObject.Builder(Directory).relations(lambda a: a.name("listings").of(Listing).me("directory")).update()
store.register(Labels)
SCHEMAS = [Int32, Size, Phone, Contact, Directory, Listing]


def entries(module):
    """A module's entries, as plain data."""
    plain = Plain.ToPlain(store)(S.Module.Schema, module)
    return plain["objects"][plain["root"]].get("schemas", [])


def module_of(*entries):
    """A module decoded from plain entries, which `Modules.module` would not write."""
    return Plain.FromPlain(B)(S.Module.Schema, {"root": "s0", "objects": {"s0": {"schemas": list(entries)}}})


def named(name):
    return {"named": {"name": name}}


STR = {"native": {"format": "basic", "token": "str"}}

## MOD-01 · The meta-schemas validate, and `Schemas.Module` is registered as a reference object schema

In [ ]:
meta = [S.Module.Schema, S.Module.Entry, S.Module.Definition, S.OfAny.Schema, S.OfAny.Named, S.OfNative.Schema,
        S.OfProperty.Schema, S.OfObject.Schema, S.OfAdjacency.Schema, S.OfRelation.Schema, S.OfRelation.Ref,
        S.OfUnion.Schema, S.OfIntersection.Schema, S.OfIndexed.Schema]
assert all(schema.validate() == [] for schema in meta)
assert Modules.MODULE == "Schemas.Module" and store.schema("Schemas.Module") is S.Module.Schema and S.Module.Schema.ref
assert store.registered("Schemas.Module") is S.Module.Schema and store.registered("Labels") is Labels
with raises(LookupError, match="no schema registered as 'nope'"):
    store.registered("nope")
kinds = ["native", "object", "union", "intersection", "indexed", "apply"]
assert [b.name for b in S.OfAny.Schema.branches] == [*kinds, "named"]  # a type: a schema of any kind, or a name
assert [b.name for b in S.Module.Definition.branches] == [*kinds, "relation"]
assert [b.name for b in S.OfRelation.Ref.branches] == ["relation", "named"]
assert list(S.OfObject.Schema.properties) == ["name", "parameters", "properties", "adjacencies", "singleton", "ref", "description"]
assert list(S.OfRelation.Schema.properties) == ["name", "parameters", "links", "properties", "uniques", "description"]
assert list(S.OfNative.Schema.properties) == ["name", "parameters", "format", "token", "bits", "bytes", "terms", "description"]

## MOD-02 · A module writes each schema inline, by kind, and refers by name to the schemas in it and those registered

In [ ]:
module = Modules.module(store, SCHEMAS)
assert module.schema_name() == "Schemas.Module" and Validators.Validate(B)(S.Module.Schema, module) == []
assert Modules.module(B, SCHEMAS) is not module
phone = {"object": {"properties": [{"name": "number", "type": STR}]}}
assert entries(module) == [
    {"name": "Int32", "schema": {"native": {"format": "basic", "token": "int", "bits": 32}}},
    {"name": "Size", "schema": {"native": {"format": "ccpp", "token": "size_t", "bytes": 8}}},
    {"name": "Phone", "schema": phone},
    {"name": "Contact", "schema": {"object": {
        "properties": [
            {"name": "name", "type": STR},
            {"name": "home", "type": named("Phone")},
            {"name": "reach", "type": {"union": {"branches": [
                {"name": "phone", "type": named("Phone")},
                {"name": "other", "type": {"object": {"properties": [{"name": "note", "type": STR}]}}}]}}},
            {"name": "stamped", "type": {"intersection": {"parts": [
                {"name": "a", "type": named("Phone")}, {"name": "b", "type": named("Phone")}]}}},
            {"name": "ids", "type": {"indexed": {"item": named("Int32")}}}],
        "adjacencies": [{"name": "listings", "relation": named("Listing"), "me": "contact"},
                        {"name": "labels", "relation": named("Labels"), "me": "contact"}],  # registered
        "ref": True}}},
    {"name": "Directory", "schema": {"object": {"adjacencies": [
        {"name": "listings", "relation": named("Listing"), "me": "directory"}], "singleton": "Directory", "ref": True}}},
    {"name": "Listing", "schema": {"relation": {"links": ["directory", "contact"], "properties": [{"name": "key", "type": STR}],
                                                "uniques": [["directory", "key"], ["contact"]]}}}]
# What has no name is written inline wherever it is used: a relation too. Empty contents are left out.
Pair = S.OfRelation.Builder().links("a", "b").create()
Lone = S.OfObject.Builder().name("Lone").relations(lambda a: a.name("pairs").of(Pair).me("a")).create()
assert entries(Modules.module(store, [Lone, S.OfUnion.Builder().name("Empty").create()])) == [
    {"name": "Lone", "schema": {"object": {"adjacencies": [
        {"name": "pairs", "relation": {"relation": {"links": ["a", "b"]}}, "me": "a"}]}}},
    {"name": "Empty", "schema": {"union": {}}}]
assert entries(Modules.module(store, [])) == [] and Modules.schemas(store, getattr(B, "Schemas.Module")().create()) == {}
# The module is an object like any other: written and read as JSON and YAML, and compared.
text_ = JSON.ToJSON(store)(S.Module.Schema, module)
assert JSON.ToJSON(store)(S.Module.Schema, JSON.FromJSON(B)(S.Module.Schema, text_)) == text_
assert YAML.ToYAML(store)(S.Module.Schema, YAML.FromYAML(B)(S.Module.Schema, YAML.ToYAML(store)(S.Module.Schema, module))) == \
    YAML.ToYAML(store)(S.Module.Schema, module)
assert C.OfObject(S.Module.Schema, module).compare(C.OfObject(S.Module.Schema, Modules.module(store, SCHEMAS))) == 0
assert C.OfObject(S.Module.Schema, module).compare(C.OfObject(S.Module.Schema, Modules.module(store, [Phone]))) is None

## MOD-03 · Reading a module back gives the same schemas, sharing what is named, recursive ones included

In [ ]:
module = Modules.module(store, SCHEMAS)
back = Modules.schemas(store, JSON.FromJSON(B)(S.Module.Schema, JSON.ToJSON(store)(S.Module.Schema, module)))
assert list(back) == [schema.name for schema in SCHEMAS] and all(all(schema is not original for original in SCHEMAS) for schema in back.values())
assert all(schema.name == name for name, schema in back.items())  # each read back under its name
assert all(schema.validate() == [] for schema in back.values())
assert JSON.ToJSON(store)(S.Module.Schema, Modules.module(store, back.values())) == JSON.ToJSON(store)(S.Module.Schema, module)
contact = back["Contact"]
assert contact.properties["home"].type is back["Phone"] and contact.properties["reach"].type.branches[0].type is back["Phone"]
assert contact.properties["stamped"].type.parts[1].type is back["Phone"] and contact.properties["ids"].type.item is back["Int32"]
assert contact.adjacencies["listings"].relation is back["Listing"] is back["Directory"].adjacencies["listings"].relation
assert contact.adjacencies["labels"].relation is Labels  # resolved in the store
assert contact.adjacencies["listings"].me == "contact" and contact.ref and back["Directory"].singleton == "Directory"
assert back["Int32"].type is int and back["Int32"].bits == 32 and back["Size"].type is None and back["Size"].bytes == 8
assert back["Listing"].links == ("directory", "contact") and back["Listing"].uniques == (
    frozenset({"directory", "key"}), frozenset({"contact"}))
# A schema that refers to itself is written once, by name, and read back as itself.
Node = S.OfObject.Builder().name("Node").properties(text("label")).create()
S.OfObject.Builder(Node).properties(lambda p: p.name("children").of(lambda t: t.as_indexed(lambda i: i.of(Node)))).update()
tree = Modules.schemas(store, Modules.module(store, [Node]))["Node"]
assert tree.properties["children"].type.item is tree and tree.validate() == []
# Validating a schema that holds itself ends, and reports each problem once: as a property, an entry's or in a union.
S.OfObject.Builder(Node).properties(lambda p: p.name("w").of(lambda t: t.as_native(lambda n: n.type(int).bits(0)))).update()
assert Node.validate() == ["property 'w': a width in bits must be a positive int or a term, got 0"]
assert S.OfRelation.Builder().links("a", "b").properties(lambda p: p.name("tree").of(tree)).create().validate() == []
Loop = S.OfUnion.Builder().create()
S.OfUnion.Builder(Loop).branches(lambda b: b.name("more").of(lambda t: t.as_indexed(lambda i: i.of(Loop))),
                                 lambda b: b.name("ints").of(lambda t: t.as_indexed(lambda i: i.of(Int32)))).update()
assert S.OfObject.Builder().properties(lambda p: p.name("loop").of(Loop)).create().validate() == []
# Keys and extents are written and read back.
Grid = S.OfObject.Builder().name("Grid").properties(
    lambda p: p.name("attrs").of(lambda t: t.as_indexed(lambda i: i.key(lambda t: t.as_native(str)).of(Int32))),
    lambda p: p.name("row").of(lambda t: t.as_indexed(lambda i: i.of(Int32).extent(1, 3))),
    lambda p: p.name("open").of(lambda t: t.as_indexed(lambda i: i.of(Int32).extent(2)))).create()
grid = Modules.schemas(store, Modules.module(store, [Grid, Int32]))["Grid"]
assert grid.properties["attrs"].type.key.type is str and grid.properties["attrs"].type.extent is None
assert grid.properties["row"].type.extent == S.OfIndexed.Extent(1, 3) and grid.properties["row"].type.key is None
assert grid.properties["open"].type.extent == S.OfIndexed.Extent(2, None)
written = Modules.schemas(store, module_of({"name": "L", "schema": {"indexed": {"item": STR, "extent": {"maximum": 3}}}}))["L"]
assert written.extent == S.OfIndexed.Extent(0, 3)  # a minimum left out is 0
# Inline schemas are read as new schemas of their kind.
Pair = S.OfRelation.Builder().links("a", "b").create()
Lone = S.OfObject.Builder().name("Lone").relations(lambda a: a.name("pairs").of(Pair).me("a")).create()
lone = Modules.schemas(store, Modules.module(store, [Lone]))["Lone"]
assert lone.adjacencies["pairs"].relation.links == ("a", "b") and lone.adjacencies["pairs"].relation is not Pair

## MOD-04 · Refused: a schema referring to itself unnamed, an unknown name, a name of the wrong kind, a name defined twice

In [ ]:
Node = S.OfObject.Builder().properties(text("label")).create()
S.OfObject.Builder(Node).properties(lambda p: p.name("next").of(lambda t: t.as_indexed(lambda i: i.of(Node)))).update()
Holder = S.OfObject.Builder().name("Holder").properties(lambda p: p.name("root").of(Node)).create()
with raises(ValueError, match="a schema that refers to itself must be named"):
    Modules.module(store, [Holder])
with raises(ValueError, match="a module holds named schemas; name each with its builder's .name()"):
    Modules.module(store, [Phone, S.OfUnion.Data()])
with raises(TypeError, match="not a schema: 'text'"):
    Modules.module(store, [S.OfObject.Data(properties={"p": S.OfProperty.Data("p", "text")}, name="Bad")])
with raises(TypeError, match="unsupported native type 'weird'"):
    Modules.module(store, [S.OfNative.Data("weird", name="Bad")])
property_named = lambda name: {"object": {"properties": [{"name": "p", "type": named(name)}]}}  # noqa: E731
with raises(LookupError, match="no schema named 'Nope' in the module or the store"):
    Modules.schemas(store, module_of({"name": "A", "schema": property_named("Nope")}))
with raises(TypeError, match="'Labels' is a relation, not a type"):
    Modules.schemas(store, module_of({"name": "A", "schema": property_named("Labels")}))
with raises(TypeError, match="'A' is not a relation"):
    Modules.schemas(store, module_of({"name": "A", "schema": {"object": {"adjacencies": [
        {"name": "x", "relation": named("A"), "me": "a"}]}}}))
with raises(ValueError, match="the module defines 'A' twice"):
    Modules.schemas(store, module_of({"name": "A", "schema": {"union": {}}}, {"name": "A", "schema": {"union": {}}}))

## MOD-05 · One type by name or inline: `reference` and `resolve`

In [ ]:
local = Proxies.OfStore()
Card = S.OfObject.Builder().name("Card").ref().properties(text("name")).create()
local.register(Card)
Note = S.OfObject.Builder().properties(text("text"), lambda p: p.name("card").of(Card)).create()
assert Modules.reference(Card) == {"named": {"name": "Card"}}  # named: by name
inline = Modules.reference(Note)  # unnamed: inline, its members by name where they have one
assert inline == {"object": {"properties": [{"name": "text", "type": {"native": {"format": "basic", "token": "str"}}},
                                            {"name": "card", "type": {"named": {"name": "Card"}}}]}}
assert Modules.resolve(local, {"named": {"name": "Card"}}) is Card
copy = Modules.resolve(local, inline)
assert copy is not Note and copy.properties["card"].type is Card and Modules.reference(copy) == inline
with raises(LookupError, match="no schema named 'Nope' in the module or the store"):
    Modules.resolve(local, {"named": {"name": "Nope"}})

## MOD-06 · Descriptions travel with a module, written only where present

In [ ]:
Meters = S.OfNative.resolve(lambda n: n.name("Meters").type(float).description("A length"))
Calls = S.OfRelation.Builder().name("Calls").description("Who called whom").links("caller", "callee").properties(
    lambda p: p.name("at").of(lambda t: t.as_native(str)).description("When")).create()
Line = S.OfObject.Builder().name("Line").ref().description("A phone line").properties(
    lambda p: p.name("length").of(Meters).description("Of cable"),
    lambda p: p.name("reach").of(S.OfUnion.Builder().description("Near or far").branches(
        lambda b: b.name("near").of(Meters).description("Within reach"), lambda b: b.name("far").of(Meters)).create()),
    lambda p: p.name("both").of(S.OfIntersection.Builder().description("Both at once").parts(
        lambda b: b.name("a").of(Phone).description("The first"), lambda b: b.name("b").of(Phone)).create()),
    lambda p: p.name("tags").of(lambda t: t.as_indexed(lambda i: i.of(Meters).description("Any number")))
    ).relations(lambda a: a.name("calls").of(Calls).me("caller").description("Calls made")).create()
module = Modules.module(store, [Phone, Meters, Calls, Line])
assert entries(module)[1:] == [  # after Phone, as in MOD-02
    {"name": "Meters", "schema": {"native": {"format": "basic", "token": "float", "description": "A length"}}},
    {"name": "Calls", "schema": {"relation": {
        "links": ["caller", "callee"], "properties": [{"name": "at", "type": STR, "description": "When"}],
        "description": "Who called whom"}}},
    {"name": "Line", "schema": {"object": {
        "properties": [
            {"name": "length", "type": named("Meters"), "description": "Of cable"},
            {"name": "reach", "type": {"union": {"branches": [
                {"name": "near", "type": named("Meters"), "description": "Within reach"},
                {"name": "far", "type": named("Meters")}], "description": "Near or far"}}},
            {"name": "both", "type": {"intersection": {"parts": [
                {"name": "a", "type": named("Phone"), "description": "The first"},
                {"name": "b", "type": named("Phone")}], "description": "Both at once"}}},
            {"name": "tags", "type": {"indexed": {"item": named("Meters"), "description": "Any number"}}}],
        "adjacencies": [{"name": "calls", "relation": named("Calls"), "me": "caller", "description": "Calls made"}],
        "ref": True, "description": "A phone line"}}}]
back = Modules.schemas(store, JSON.FromJSON(B)(S.Module.Schema, JSON.ToJSON(store)(S.Module.Schema, module)))
assert JSON.ToJSON(store)(S.Module.Schema, Modules.module(store, back.values())) == JSON.ToJSON(store)(S.Module.Schema, module)
line = back["Line"]
assert back["Meters"] == Meters and back["Calls"].description == "Who called whom" and line.description == "A phone line"
assert back["Calls"].properties["at"].description == "When" and line.properties["length"].description == "Of cable"
assert line.properties["reach"].type.properties["near"].description == "Within reach" and line.adjacencies["calls"].description == "Calls made"
assert line.properties["both"].type.description == "Both at once" and line.properties["tags"].type.description == "Any number"
assert all(schema.validate() == [] for schema in back.values())